In [72]:
import json, pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report

# === 1) 데이터 적재 & Aspects 폭발 ===
path = "data/패션/1-1.여성의류(196).json"
df = pd.read_json(path)
df

,Index,RawText,Source,Domain,MainCategory,ProductName,Syllable,Word,GeneralPolarity,Aspects
0,1024338,안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 ...,SNS,패션,여성의류,OO 경량 다운 자켓,513,121,1,"[{'Aspect': '디자인', 'SentimentText': '딱 기본 스타일인..."
1,1024477,드디어 겨울이 찾아왔네요. 이제부터 슬슬 겨울 패딩 장만하셔야지요? 패딩 소개해 드...,SNS,패션,여성의류,OO 아** 구스코트,464,105,1,"[{'Aspect': '사이즈', 'SentimentText': '저는 블랙90 사..."
2,1025044,오늘도 정말 춥네요... 롱패딩 찾고 계신 분을 위한 후기 공유합니다. 키 158...,SNS,패션,여성의류,OO 아** 구스코트,314,78,1,"[{'Aspect': '사이즈', 'SentimentText': '키 158센티로에..."
3,1025046,이웃님들 오늘도 안녕하신가요? 오늘은 따끈한 신상 패딩 후기 올려봅니다~~ 겨울이...,SNS,패션,여성의류,OO 아** 구스코트,307,74,1,"[{'Aspect': '색상', 'SentimentText': '흰색 패딩이 너무나..."
4,1025071,OOO 구스로 소문난 OO의 롱패딩~ 한번 구경 가봐요. 일단 보는 순간 고급스럽...,SNS,패션,여성의류,OO 아** 구스코트,279,68,1,"[{'Aspect': '소재', 'SentimentText': ' 일단 보는 순간 ..."
...,...,...,...,...,...,...,...,...,...,...
118,1028153,요즘 출 퇴근할 때 너무 추워서 목폴라 상품을 보고 있다가 마음에 드는 옷을 발견했...,SNS,패션,여성의류,OO 여성용 목폴라 티셔츠,294,75,1,"[{'Aspect': '두께', 'SentimentText': '저는 두께가 얇은 ..."
119,1028154,오늘 소개 해 드릴 옷은 겨울에 따뜻하게 입을 수 있는 원피스 하나 소개 해 드리려...,SNS,패션,여성의류,OO 케** 반집업니트원피스,296,75,1,"[{'Aspect': '기능', 'SentimentText': '겨울에 따뜻하게 입..."
120,1028155,기존 목폴라 티셔츠가 다 늘어져 구입하려고 찾아보던 중 알게 된 OOO 비네츠 폴라...,SNS,패션,여성의류,OO 비** 폴라티,298,71,1,"[{'Aspect': '소재', 'SentimentText': '이 목폴라 티셔츠는..."
121,1028156,몸에 딱 맞는 바지 찾기가 너무 힘든데 제가 원하던 바지를 찾아서 글을 남깁니다. ...,SNS,패션,여성의류,OO 프리미엄 팬츠,318,78,1,"[{'Aspect': '소재', 'SentimentText': '일단 원단이 너무 ..."


In [73]:

rows = []
for _, r in df.iterrows():
    aspects = r.get("Aspects", []) or []
    for a in aspects:
        stext = a.get("SentimentText")
        asp   = a.get("Aspect")
        pol   = a.get("SentimentPolarity")
        if stext and asp is not None and pol is not None:
            rows.append({"text": stext, "aspect": str(asp), "polarity": int(pol)})

data = pd.DataFrame(rows).dropna()
# (선택) 감성값 범위 제한
data = data[data["polarity"].isin([-1,0,1])]
data

,text,aspect,polarity
0,딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서,디자인,1
1,이것만 입기엔 얇지만,두께,-1
2,초겨울까지는 운동 갈때 안에 얇은 기능성 반팔 입고 요것만 입어도 꽤 따뜻해요.,기능,1
3,색상도 디자인도 무난해서,색상,0
4,디자인도 무난해서,디자인,0
...,...,...,...
912,길이감도 너무 짧거나 애매한 길이가 아니고 적당합니다.,길이,1
913,트레이닝 세트나 데님 스커트 후드 원피스 등에도 잘 어울립니다.,활용성,1
914,전체적으로 고급스러움이 잘 녹아있는 디자인으로 되어 있고,디자인,1
915,퀄리티도 넘 휼륭합니다.,품질,1


In [74]:

# === 2) 라벨 인코딩 ===
le_aspect = LabelEncoder()
y_aspect  = le_aspect.fit_transform(data["aspect"])

# polarity는 -1/0/1 그대로 써도 되지만, 분류기 호환을 위해 인코딩
le_pol = LabelEncoder()
y_polar = le_pol.fit_transform(data["polarity"])

X = data["text"].astype(str)
X

0        딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서
1                                       이것만 입기엔 얇지만
2      초겨울까지는 운동 갈때 안에 얇은 기능성 반팔 입고 요것만 입어도 꽤 따뜻해요.
3                                     색상도 디자인도 무난해서
4                                         디자인도 무난해서
                           ...                     
912                  길이감도 너무 짧거나 애매한 길이가 아니고 적당합니다.
913           트레이닝 세트나 데님 스커트 후드 원피스 등에도 잘 어울립니다.  
914                 전체적으로 고급스러움이 잘 녹아있는 디자인으로 되어 있고
915                                   퀄리티도 넘 휼륭합니다.
916               그냥 툭툭 걸치기만 해도 스타일이 멋스러운 숏패딩이었습니다.
Name: text, Length: 917, dtype: object

In [75]:

# === 3) stratify: Aspect×Polarity 결합키로 층화 분할 ===
X_tr, X_te, ya_tr, ya_te, yp_tr, yp_te = train_test_split(
    X, y_aspect, y_polar, test_size=0.2, random_state=42
)


In [76]:

# === 4) 토크나이저(옵션: Komoran) ===
from konlpy.tag import Komoran
komoran = Komoran()
def tokenize_komoran(s):
    return [t for t in komoran.morphs(s) if len(t) > 1]

# === 5) 파이프라인: TF-IDF -> (로지스틱 멀티아웃풋) ===
vec = TfidfVectorizer(
    tokenizer=tokenize_komoran,  # Komoran 사용할 때 주석 해제
    min_df=2, max_df=0.9, ngram_range=(1,2)
)

base_clf = LogisticRegression(
    max_iter=1000, class_weight="balanced", n_jobs=None
)

clf = MultiOutputClassifier(base_clf)  # [aspect_classifier, polarity_classifier]처럼 동작

pipe = Pipeline([
    ("tfidf", vec),
    ("clf", clf)
])

pipe.fit(X_tr, np.column_stack([ya_tr, yp_tr]))

pred = pipe.predict(X_te)
pred_aspect  = pred[:,0]
pred_polar   = pred[:,1]


c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [77]:

print("=== ASPECT ===")
print(classification_report(ya_te, pred_aspect, target_names=le_aspect.classes_))


=== ASPECT ===
              precision    recall  f1-score   support

          가격       1.00      1.00      1.00        19
          기능       0.74      0.68      0.71        25
          길이       0.38      1.00      0.55         3
          두께       0.75      1.00      0.86         3
         디자인       1.00      0.69      0.81        32
          마감       0.33      1.00      0.50         1
          무게       1.00      0.83      0.91         6
         사이즈       0.82      1.00      0.90         9
          색상       0.73      0.85      0.79        13
          소재       0.91      0.62      0.74        16
         신축성       1.00      1.00      1.00         1
        제품구성       0.33      0.25      0.29         4
         착용감       0.70      0.88      0.78         8
          촉감       1.00      0.83      0.91         6
          품질       0.50      0.33      0.40         6
           핏       0.67      0.67      0.67         9
         활용성       0.65      0.87      0.74        23

    accurac

In [78]:

print("=== POLARITY ===")
print(classification_report(yp_te, pred_polar, target_names=[str(c) for c in le_pol.classes_]))


=== POLARITY ===
              precision    recall  f1-score   support

          -1       0.30      0.38      0.33        16
           0       0.00      0.00      0.00         3
           1       0.92      0.88      0.90       165

    accuracy                           0.83       184
   macro avg       0.41      0.42      0.41       184
weighted avg       0.85      0.83      0.84       184



In [79]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

# (위에서 만든 data 재사용)
data["combo"] = data["aspect"].astype(str) + "__" + data["polarity"].astype(str)

y_combo = data["combo"]
X = data["text"].astype(str)
y_combo

0      디자인__1
1      두께__-1
2       기능__1
3       색상__0
4      디자인__0
        ...  
912     길이__1
913    활용성__1
914    디자인__1
915     품질__1
916    디자인__1
Name: combo, Length: 917, dtype: object

In [81]:

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_combo, test_size=0.2, random_state=42
)

pipe2 = Pipeline([
    ("tfidf", TfidfVectorizer(min_df=2, max_df=0.9, ngram_range=(1,2))),
    ("clf", LinearSVC())  # 속도 빠르고 멀티클래스 One-vs-Rest 자동
])

pipe2.fit(X_tr, y_tr)
pred2 = pipe2.predict(X_te)

print(accuracy_score(y_te, pred2))
print(f1_score(y_te, pred2, average='macro'))



0.4891304347826087
0.27330553012051356


In [82]:
import pandas as pd, numpy as np, re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score, classification_report
from difflib import SequenceMatcher

# 0) 문장 분할기: kss가 있으면 kss, 없으면 간단한 정규식 대체
try:
    import kss
    def sent_split(text: str):
        return [s.strip() for s in kss.split_sentences(text) if s.strip()]
except Exception:
    _kr_end = re.compile(r'(?<=[.!?])\s+|(?<=[다요까네죠임음임니까습니다]|[?!])\s+')
    def sent_split(text: str):
        return [s.strip() for s in _kr_end.split(text) if s.strip()]

# 1) 데이터 로드
path = "data/패션/1-1.여성의류(196).json"
raw = pd.read_json(path)


In [83]:

rows = []
for ridx, r in raw.iterrows():
    rt = str(r.get("RawText") or "").strip()
    if not rt:
        continue
    sents = sent_split(rt)
    if not sents:
        continue

    # 문장 행 생성(초기 라벨은 공집합/없음)
    for sid, s in enumerate(sents):
        rows.append({
            "review_id": ridx, "sent_id": sid, "text": s,
        })

sent_df = pd.DataFrame(rows)
sent_df

,review_id,sent_id,text
0,0,0,안녕하세요
1,0,1,이웃님들 반갑습니다.
2,0,2,요즘 날씨가 많이 쌀쌀해졌죠?
3,0,3,요즘 계절에 입으면 딱 좋을 인생 경량 패딩 하나 소개해 드릴게요.
4,0,4,다 함께 go go go~~ 제가 입어보고 좋아서 엄마께도 구매해드렸네요.
...,...,...,...
1230,122,4,가벼우면서도 보온성이 좋습니다.
1231,122,5,길이감도 너무 짧거나 애매한 길이가 아니고 적당합니다.
1232,122,6,트레이닝 세트나 데님 스커트 후드 원피스 등에도 잘 어울립니다.
1233,122,7,전체적으로 고급스러움이 잘 녹아있는 디자인으로 되어 있고 퀄리티도 넘 휼륭합니다.


In [84]:
pred = pipe.predict(sent_df['text'])

In [85]:
sent_df[['aspect', 'pola']] = pred

In [86]:
sent_df['aspect'] = le_aspect.inverse_transform(sent_df['aspect'])

sent_df['pola'] = le_pol.inverse_transform(sent_df['pola'])



In [87]:
sent_df['pola'].value_counts()

pola
 1    1009
-1     167
 0      59
Name: count, dtype: int64